In [0]:
import copy
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

# Task fields that hold a file path which may be relative to base_path
_PATH_FIELDS = [
    ("notebook_task", "notebook_path"),
    ("spark_python_task", "python_file"),
    ("sql_task", "file", "path"),
]

def _notebook_dir():
    """Workspace folder of the notebook this code is running in."""
    path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    return "/Workspace" + os.path.dirname(path)

def _resolve(path, base_path):
    """Leave absolute paths and URIs (dbfs:/, s3://, ...) alone; prefix relative ones."""
    if path.startswith("/") or ":/" in path:
        return path
    return f"{base_path}/{path}"

def _build_task(t, base_path):
    spec = {k: v for k, v in copy.deepcopy(t).items() if v is not None}
    if "task_name" in spec:
        spec["task_key"] = spec.pop("task_name")

    # Shorthand: 'file_path' alone means a notebook task relative to base_path
    if "file_path" in spec:
        spec["notebook_task"] = {"notebook_path": spec.pop("file_path").lstrip("/")}

    if not any(k.endswith("_task") for k in spec):
        raise ValueError(f"Task '{spec.get('task_key')}' has no task type (e.g. notebook_task, sql_task).")

    for *parents, leaf in _PATH_FIELDS:
        node = spec
        for p in parents:
            node = node.get(p) if isinstance(node, dict) else None
        if isinstance(node, dict) and node.get(leaf):
            node[leaf] = _resolve(node[leaf], base_path)

    return jobs.Task.from_dict(spec)

def create_job_config(job_name, job_tasks, job_parameters=None, base_path=None, replace=False, **job_settings):
    """Create a Lakeflow job from a list of task dicts.

    job_tasks:      list of dicts. Each needs 'task_name' plus either
                      - 'file_path' (shorthand for a notebook task), or
                      - any Jobs API task type: 'notebook_task', 'spark_python_task',
                        'python_wheel_task', 'sql_task', 'pipeline_task', 'dbt_task',
                        'run_job_task', 'condition_task', 'for_each_task', ...
                    Any other Jobs API task field passes through as-is
                    ('depends_on', 'environment_key', 'job_cluster_key', 'libraries',
                    'max_retries', 'run_if', ...).
    job_parameters: [{'key': ..., 'value': ...}]
    base_path:      folder that relative file paths resolve against (default: this notebook's folder)
    replace:        if True, delete any existing job with the same name first
    job_settings:   extra arguments for w.jobs.create (environments, job_clusters, schedule, tags, ...)
    """
    existing = list(w.jobs.list(name=job_name))
    if existing:
        if not replace:
            raise ValueError(f"Job '{job_name}' already exists (job_id={existing[0].job_id}).")
        for j in existing:
            w.jobs.delete(job_id=j.job_id)
            print(f"Deleted existing job {j.job_id}")
    print(f"Job name is unique. Creating the job {job_name}...")

    base_path = (base_path or _notebook_dir()).rstrip("/")
    print(f"Using the following path to reference the Files: {base_path}/")

    job = w.jobs.create(
        name=job_name,
        tasks=[_build_task(t, base_path) for t in job_tasks],
        parameters=[
            jobs.JobParameterDefinition(name=p["key"], default=p["value"])
            for p in (job_parameters or [])
        ],
        **job_settings,
    )
    print(f"Created job_id={job.job_id}")
    return job.job_id

In [0]:
# # Dicts to add any kind of tasks to a job through the databricks SDK

# # --- CODE TASKS ---
# # Notebook (full form, with parameters)
# {"task_name": "ingest",
#  "notebook_task": {"notebook_path": "01_ingest",
#                    "base_parameters": {"source": "sales"}}}

# # Python script
# {"task_name": "transform",
#  "spark_python_task": {"python_file": "src/transform.py",
#                        "parameters": ["--env", "dev"]},
#  "environment_key": "default"}

# # Python wheel
# {"task_name": "wheel_step",
#  "python_wheel_task": {"package_name": "my_pkg",
#                        "entry_point": "main",
#                        "named_parameters": {"env": "dev"}},
#  "environment_key": "default"}   # the wheel goes in that environment's dependencies

# # JAR (classic compute)
# {"task_name": "jar_step",
#  "spark_jar_task": {"main_class_name": "com.acme.Main",
#                     "parameters": ["dev"]},
#  "libraries": [{"jar": "/Volumes/main/libs/app.jar"}],
#  "job_cluster_key": "main_cluster"}


#  # --- SQL AND DBT TASKS ---
#  # SQL file
# {"task_name": "report",
#  "sql_task": {"file": {"path": "sql/report.sql"},
#               "warehouse_id": "abc123",
#               "parameters": {"run_date": "2026-11-11"}}}

# # Saved SQL query
# {"task_name": "saved_query",
#  "sql_task": {"query": {"query_id": "1111-2222"},
#               "warehouse_id": "abc123"}}

# # SQL alert
# {"task_name": "alert",
#  "sql_task": {"alert": {"alert_id": "3333-4444"},
#               "warehouse_id": "abc123"}}

# # dbt
# {"task_name": "dbt_build",
#  "dbt_task": {"project_directory": "/Workspace/Users/you/dbt_project",
#               "commands": ["dbt deps", "dbt build"],
#               "warehouse_id": "abc123",
#               "catalog": "dev",
#               "schema": "analytics"},
#  "environment_key": "dbt_env"}   # environment with dbt-databricks as a dependency


#  # --- ORCHESTRATION TASKS ---
#  # Lakeflow pipeline
# {"task_name": "refresh_pipeline",
#  "pipeline_task": {"pipeline_id": "1234-abcd", "full_refresh": False}}

# # Run another job
# {"task_name": "trigger_downstream",
#  "run_job_task": {"job_id": 987654,
#                   "job_parameters": {"catalog": "dev"}}}

# # If/else condition
# {"task_name": "check_rows",
#  "condition_task": {"op": "GREATER_THAN",
#                     "left": "{{tasks.ingest.values.row_count}}",
#                     "right": "0"},
#  "depends_on": [{"task_key": "ingest"}]}

# # Task that runs only on the "true" branch
# {"task_name": "load",
#  "file_path": "02_load",
#  "depends_on": [{"task_key": "check_rows", "outcome": "true"}]}

# # For each (loop)
# {"task_name": "per_country",
#  "for_each_task": {"inputs": '["CO", "MX", "PE"]',
#                    "concurrency": 2,
#                    "task": {"task_key": "per_country_iteration",
#                             "notebook_task": {
#                                 "notebook_path": "/Workspace/Users/you/project/03_country",
#                                 "base_parameters": {"country": "{{input}}"}}}}}

# # AI/BI dashboard refresh
# {"task_name": "refresh_dashboard",
#  "dashboard_task": {"dashboard_id": "5555-6666", "warehouse_id": "abc123"}}


# # --- SETTINGS THAT WORK ON ANY TASKS ---
# {"task_name": "robust_step",
#  "file_path": "04_step",
#  "depends_on": [{"task_key": "ingest"}, {"task_key": "transform"}],
#  "run_if": "ALL_DONE",               # run even if upstream failed
#  "max_retries": 2,
#  "min_retry_interval_millis": 60000,
#  "timeout_seconds": 3600}


# # --- COMPUTE DEFINITIONS ---
# create_job_config(
#     job_name="demo_job",
#     job_tasks=[...],
#     environments=[
#         {"environment_key": "default",
#          "spec": {"environment_version": "2",
#                   "dependencies": ["/Volumes/main/libs/my_pkg-0.1-py3-none-any.whl"]}},
#         {"environment_key": "dbt_env",
#          "spec": {"environment_version": "2", "dependencies": ["dbt-databricks"]}},
#     ],
#     job_clusters=[
#         {"job_cluster_key": "main_cluster",
#          "new_cluster": {"spark_version": "15.4.x-scala2.12",
#                          "node_type_id": "Standard_DS3_v2",
#                          "num_workers": 2}},
#     ],
# )

---

In [0]:
# --- Def to create a job config version 1 ---

# import os
# from databricks.sdk import WorkspaceClient
# from databricks.sdk.service import jobs

# w = WorkspaceClient()

# def _notebook_dir():
#     """Workspace folder of the notebook this code is running in."""
#     path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
#     return "/Workspace" + os.path.dirname(path)

# def create_job_config(job_name, job_tasks, job_parameters=None, base_path=None, replace=False):
#     """Create a Lakeflow job from a list of task dicts.

#     job_tasks:      [{'task_name': ..., 'file_path': ..., 'depends_on': None | [{'task_key': ...}]}]
#     job_parameters: [{'name': ..., 'default': ...}]
#     base_path:      folder that each file_path is relative to (default: this notebook's folder)
#     replace:        if True, delete any existing job with the same name first
#     """
#     existing = list(w.jobs.list(name=job_name))
#     if existing:
#         if not replace:
#             raise ValueError(f"Job '{job_name}' already exists (job_id={existing[0].job_id}).")
#         for j in existing:
#             w.jobs.delete(job_id=j.job_id)
#             print(f"Deleted existing job {j.job_id}")
#     print(f"Job name is unique. Creating the job {job_name}...")

#     base_path = (base_path or _notebook_dir()).rstrip("/")
#     print(f"Using the following path to reference the Files: {base_path}/")

#     tasks = [
#         jobs.Task(
#             task_key=t["task_name"],
#             notebook_task=jobs.NotebookTask(
#                 notebook_path=f"{base_path}/{t['file_path'].lstrip('/')}"
#             ),
#             depends_on=[
#                 jobs.TaskDependency(task_key=d["task_key"])
#                 for d in (t.get("depends_on") or [])
#             ] or None,
#         )
#         for t in job_tasks
#     ]

#     job = w.jobs.create(
#         name=job_name,
#         tasks=tasks,
#         parameters=[
#             jobs.JobParameterDefinition(name=p["name"], default=p["default"])
#             for p in (job_parameters or [])
#         ],
#     )
#     print(f"Created job_id={job.job_id}")
#     return job.job_id